# K-Nearest Neighbor Classifiers for the Penguins Dataset
## k Selected via Cross-Validation

This notebook implements two KNN classifiers on the Palmer Penguins dataset:
- **Classifier 1**: Predict **species** (Adelie, Chinstrap, Gentoo)
- **Classifier 2**: Predict **sex** (Male, Female)

The optimal value of **k** is chosen independently for each classifier by sweeping a range of k values and selecting the one that maximizes stratified cross-validation accuracy. Both classifiers are then evaluated on a held-out test set.

## 1. Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("All libraries imported successfully.")

## 2. Load and Preprocess the Dataset

In [ ]:
df_raw = sns.load_dataset("penguins")

# Drop the 11 rows that contain NaN values
df = df_raw.dropna().reset_index(drop=True)
print(f"Rows after dropping NaNs: {len(df)}  (removed {len(df_raw) - len(df)} rows)")

# Encode categorical columns
le_island  = LabelEncoder()
le_sex     = LabelEncoder()
le_species = LabelEncoder()

df["island_enc"]  = le_island.fit_transform(df["island"])
df["sex_enc"]     = le_sex.fit_transform(df["sex"])
df["species_enc"] = le_species.fit_transform(df["species"])

print(f"\nIsland  encoding : {dict(zip(le_island.classes_,  le_island.transform(le_island.classes_)))}")
print(f"Sex     encoding : {dict(zip(le_sex.classes_,     le_sex.transform(le_sex.classes_)))}")
print(f"Species encoding : {dict(zip(le_species.classes_, le_species.transform(le_species.classes_)))}")

# ---------------------------------------------------------------
# KNN is distance-based → features MUST be standardized so that
# body mass (grams) does not dominate bill length (mm).
# Scaling is embedded in a Pipeline fitted only on training data.
# ---------------------------------------------------------------

FEAT_SPECIES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "island_enc"]
FEAT_SEX     = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g",
                "island_enc", "species_enc"]

print(f"\nFeatures — Species : {FEAT_SPECIES}")
print(f"Features — Sex     : {FEAT_SEX}")
df[FEAT_SPECIES].describe()

## 3. Helper: k Selection via Cross-Validation

The cell below defines a reusable function that:
1. Sweeps k = 1 … K_MAX on the **training split only** (no test data leakage).
2. Evaluates each k with stratified 5-fold CV inside a `StandardScaler → KNN` pipeline.
3. Plots CV mean accuracy ± 1 std across the k sweep.
4. Returns the best k and the corresponding fitted pipeline.

In [ ]:
def select_k_and_train(X_train, y_train, X_full, y_full,
                        label: str, color: str,
                        k_max: int = 30, cv_folds: int = 5):
    """
    Sweep k=1..k_max using stratified CV on the training set, pick the best k,
    refit a StandardScaler+KNN pipeline on the full training set, and plot results.

    Parameters
    ----------
    X_train, y_train : training split (used for CV sweep)
    X_full,  y_full  : full dataset (used for final CV score reported in summary)
    label            : classifier name (for plot titles)
    color            : matplotlib color string for the CV curve
    k_max            : highest k to evaluate
    cv_folds         : number of stratified folds

    Returns
    -------
    best_k      : int
    pipeline    : fitted Pipeline(StandardScaler, KNeighborsClassifier)
    cv_means    : array of mean CV accuracies for k=1..k_max
    cv_stds     : array of std  CV accuracies for k=1..k_max
    """
    skf = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=RANDOM_STATE)
    k_range  = np.arange(1, k_max + 1)
    cv_means = np.zeros(k_max)
    cv_stds  = np.zeros(k_max)

    for i, k in enumerate(k_range):
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("knn",    KNeighborsClassifier(n_neighbors=k, metric="euclidean"))
        ])
        scores = cross_val_score(pipe, X_train, y_train, cv=skf, scoring="accuracy")
        cv_means[i] = scores.mean()
        cv_stds[i]  = scores.std()

    best_idx = int(np.argmax(cv_means))
    best_k   = k_range[best_idx]

    # --- Plot CV accuracy vs k ---
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(k_range, cv_means, marker="o", color=color, linewidth=1.8, label="CV mean accuracy")
    ax.fill_between(k_range,
                    cv_means - cv_stds,
                    cv_means + cv_stds,
                    alpha=0.2, color=color, label="±1 std")
    ax.axvline(best_k, color="red", linestyle="--", linewidth=1.5,
               label=f"Best k = {best_k}  (acc = {cv_means[best_idx]:.4f})")
    ax.set_xlabel("k (number of neighbors)", fontsize=12)
    ax.set_ylabel(f"{cv_folds}-Fold CV Accuracy", fontsize=12)
    ax.set_title(f"k Selection via Cross-Validation — {label} Classifier", fontsize=13, fontweight="bold")
    ax.set_xticks(k_range)
    ax.legend(fontsize=10)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f"[{label}] Best k = {best_k}  |  CV Accuracy = {cv_means[best_idx]:.4f} ± {cv_stds[best_idx]:.4f}")

    # Refit on entire training set with the chosen k
    best_pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("knn",    KNeighborsClassifier(n_neighbors=best_k, metric="euclidean"))
    ])
    best_pipe.fit(X_train, y_train)

    return best_k, best_pipe, cv_means, cv_stds

print("Helper function defined.")

## 4. Classifier 1 — Predict Species

### 4.1 Train/Test Split

In [ ]:
X_sp = df[FEAT_SPECIES]
y_sp = df["species_enc"]

X_train_sp, X_test_sp, y_train_sp, y_test_sp = train_test_split(
    X_sp, y_sp, test_size=0.20, random_state=RANDOM_STATE, stratify=y_sp
)

print(f"Species — Training : {len(X_train_sp)}  |  Test : {len(X_test_sp)}")
print(f"Class counts (train): {pd.Series(y_train_sp).value_counts().to_dict()}")

### 4.2 k Selection via Cross-Validation — Species

In [ ]:
best_k_sp, pipe_sp, cv_means_sp, cv_stds_sp = select_k_and_train(
    X_train_sp, y_train_sp,
    X_sp, y_sp,
    label="Species",
    color="steelblue",
    k_max=30,
    cv_folds=5
)

### 4.3 Decision Boundary Visualization — Species

To visualize the learned decision boundary in 2-D, we project onto the two most informative features
(`flipper_length_mm` vs `bill_length_mm`) and refit a KNN with the chosen k on just those two scaled features.

In [ ]:
SPECIES_NAMES = le_species.classes_.tolist()
SEX_NAMES     = le_sex.classes_.tolist()

def plot_decision_boundary_2d(X_df, y, feat_x, feat_y, k, class_names,
                               title, cmap="tab10", h=0.02):
    """Fit a 2-feature StandardScaler+KNN and plot the decision boundary."""
    X2 = X_df[[feat_x, feat_y]].values
    scaler2 = StandardScaler()
    X2s = scaler2.fit_transform(X2)

    knn2 = KNeighborsClassifier(n_neighbors=k, metric="euclidean")
    knn2.fit(X2s, y)

    x_min, x_max = X2s[:, 0].min() - 0.5, X2s[:, 0].max() + 0.5
    y_min, y_max = X2s[:, 1].min() - 0.5, X2s[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = knn2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    palette = plt.get_cmap(cmap)
    n_classes = len(class_names)

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.contourf(xx, yy, Z, alpha=0.25, cmap=cmap,
                levels=np.arange(-0.5, n_classes, 1))
    scatter = ax.scatter(X2s[:, 0], X2s[:, 1],
                         c=y.values, cmap=cmap, edgecolors="k",
                         s=55, linewidths=0.6,
                         vmin=0, vmax=n_classes - 1)
    handles = [plt.Line2D([0], [0], marker="o", color="w",
                           markerfacecolor=palette(i / max(n_classes - 1, 1)),
                           markeredgecolor="k", markersize=9, label=cls)
               for i, cls in enumerate(class_names)]
    ax.legend(handles=handles, title="Class", fontsize=10)
    ax.set_xlabel(f"{feat_x} (standardized)", fontsize=11)
    ax.set_ylabel(f"{feat_y} (standardized)", fontsize=11)
    ax.set_title(title, fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

plot_decision_boundary_2d(
    df[FEAT_SPECIES], y_sp,
    feat_x="flipper_length_mm", feat_y="bill_length_mm",
    k=best_k_sp, class_names=SPECIES_NAMES,
    title=f"KNN Decision Boundary — Species (k={best_k_sp}, 2-feature projection)"
)

### 4.4 Performance Evaluation — Species

In [ ]:
y_pred_sp = pipe_sp.predict(X_test_sp)

train_acc_sp = accuracy_score(y_train_sp, pipe_sp.predict(X_train_sp))
test_acc_sp  = accuracy_score(y_test_sp,  y_pred_sp)

# Final CV score on the full dataset with the chosen k (informational)
skf_final = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
full_cv_sp = cross_val_score(pipe_sp, X_sp, y_sp, cv=skf_final, scoring="accuracy")

print("=" * 50)
print(f"  Species KNN  (k = {best_k_sp})")
print("=" * 50)
print(f"  Train Accuracy      : {train_acc_sp:.4f}")
print(f"  Test  Accuracy      : {test_acc_sp:.4f}")
print(f"  Full-dataset 5-Fold : {full_cv_sp.mean():.4f} ± {full_cv_sp.std():.4f}")
print()
print("Classification Report (test set):\n")
print(classification_report(y_test_sp, y_pred_sp, target_names=SPECIES_NAMES))

# Confusion matrix
cm_sp = confusion_matrix(y_test_sp, y_pred_sp)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm_sp, display_labels=SPECIES_NAMES).plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title(f"Confusion Matrix — Species KNN (k={best_k_sp})", fontsize=13)
plt.tight_layout()
plt.show()

## 5. Classifier 2 — Predict Sex

### 5.1 Train/Test Split

In [ ]:
X_sx = df[FEAT_SEX]
y_sx = df["sex_enc"]

X_train_sx, X_test_sx, y_train_sx, y_test_sx = train_test_split(
    X_sx, y_sx, test_size=0.20, random_state=RANDOM_STATE, stratify=y_sx
)

print(f"Sex — Training : {len(X_train_sx)}  |  Test : {len(X_test_sx)}")
print(f"Class counts (train): {pd.Series(y_train_sx).value_counts().to_dict()}")

### 5.2 k Selection via Cross-Validation — Sex

In [ ]:
best_k_sx, pipe_sx, cv_means_sx, cv_stds_sx = select_k_and_train(
    X_train_sx, y_train_sx,
    X_sx, y_sx,
    label="Sex",
    color="orchid",
    k_max=30,
    cv_folds=5
)

### 5.3 Decision Boundary Visualization — Sex

Projected onto `body_mass_g` vs `bill_depth_mm` — the two features that most strongly separate male and female penguins.

In [ ]:
plot_decision_boundary_2d(
    df[FEAT_SEX], y_sx,
    feat_x="body_mass_g", feat_y="bill_depth_mm",
    k=best_k_sx, class_names=SEX_NAMES,
    title=f"KNN Decision Boundary — Sex (k={best_k_sx}, 2-feature projection)",
    cmap="PiYG"
)

### 5.4 Performance Evaluation — Sex

In [ ]:
y_pred_sx = pipe_sx.predict(X_test_sx)

train_acc_sx = accuracy_score(y_train_sx, pipe_sx.predict(X_train_sx))
test_acc_sx  = accuracy_score(y_test_sx,  y_pred_sx)

full_cv_sx = cross_val_score(pipe_sx, X_sx, y_sx, cv=skf_final, scoring="accuracy")

print("=" * 50)
print(f"  Sex KNN  (k = {best_k_sx})")
print("=" * 50)
print(f"  Train Accuracy      : {train_acc_sx:.4f}")
print(f"  Test  Accuracy      : {test_acc_sx:.4f}")
print(f"  Full-dataset 5-Fold : {full_cv_sx.mean():.4f} ± {full_cv_sx.std():.4f}")
print()
print("Classification Report (test set):\n")
print(classification_report(y_test_sx, y_pred_sx, target_names=SEX_NAMES))

cm_sx = confusion_matrix(y_test_sx, y_pred_sx)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm_sx, display_labels=SEX_NAMES).plot(ax=ax, colorbar=False, cmap="Purples")
ax.set_title(f"Confusion Matrix — Sex KNN (k={best_k_sx})", fontsize=13)
plt.tight_layout()
plt.show()

## 6. Side-by-Side Performance Summary

In [ ]:
summary = pd.DataFrame({
    "Classifier"           : ["Species (3-class)", "Sex (binary)"],
    "Best k (CV)"          : [best_k_sp, best_k_sx],
    "Train Accuracy"       : [f"{train_acc_sp:.4f}", f"{train_acc_sx:.4f}"],
    "Test Accuracy"        : [f"{test_acc_sp:.4f}",  f"{test_acc_sx:.4f}"],
    "5-Fold CV (full data)": [f"{full_cv_sp.mean():.4f} ± {full_cv_sp.std():.4f}",
                              f"{full_cv_sx.mean():.4f} ± {full_cv_sx.std():.4f}"],
}).set_index("Classifier")

print(summary.to_string())

# --- Grouped bar chart ---
fig, ax = plt.subplots(figsize=(9, 4))
metrics  = ["Train Accuracy", "Test Accuracy", "CV Mean (full)"]
sp_vals  = [train_acc_sp, test_acc_sp, full_cv_sp.mean()]
sx_vals  = [train_acc_sx, test_acc_sx, full_cv_sx.mean()]
sp_errs  = [0, 0, full_cv_sp.std()]
sx_errs  = [0, 0, full_cv_sx.std()]

x = np.arange(len(metrics))
w = 0.35
b1 = ax.bar(x - w/2, sp_vals, w, yerr=sp_errs, capsize=4,
            label=f"Species (k={best_k_sp})", color="steelblue", edgecolor="black")
b2 = ax.bar(x + w/2, sx_vals, w, yerr=sx_errs, capsize=4,
            label=f"Sex (k={best_k_sx})", color="orchid", edgecolor="black")

ax.set_ylim(0.70, 1.05)
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_ylabel("Accuracy")
ax.set_title("KNN Performance Comparison: Species vs. Sex Classifier", fontsize=13)
ax.legend()

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.004,
            f"{bar.get_height():.3f}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

## 7. CV Curve Comparison — Both Classifiers

Overlay the k-sweep curves for direct comparison of how each task's accuracy varies with neighborhood size.

In [ ]:
k_range = np.arange(1, 31)

fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(k_range, cv_means_sp, marker="o", color="steelblue", linewidth=1.8,
        label=f"Species (best k={best_k_sp})")
ax.fill_between(k_range, cv_means_sp - cv_stds_sp, cv_means_sp + cv_stds_sp,
                alpha=0.15, color="steelblue")

ax.plot(k_range, cv_means_sx, marker="s", color="orchid", linewidth=1.8,
        label=f"Sex     (best k={best_k_sx})")
ax.fill_between(k_range, cv_means_sx - cv_stds_sx, cv_means_sx + cv_stds_sx,
                alpha=0.15, color="orchid")

ax.axvline(best_k_sp, color="steelblue", linestyle="--", linewidth=1.2, alpha=0.8)
ax.axvline(best_k_sx, color="orchid",    linestyle="--", linewidth=1.2, alpha=0.8)

ax.set_xlabel("k (number of neighbors)", fontsize=12)
ax.set_ylabel("5-Fold CV Accuracy (training set)", fontsize=12)
ax.set_title("CV Accuracy vs k — Species and Sex Classifiers", fontsize=13, fontweight="bold")
ax.set_xticks(k_range)
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Summary and Observations

### Why StandardScaler is required for KNN
KNN assigns class labels by majority vote among the k nearest neighbors, measured by Euclidean distance.
Without scaling, `body_mass_g` (range ~2700–6300 g) would dominate `bill_depth_mm` (range ~13–21 mm)
simply because its absolute values are hundreds of times larger. StandardScaler transforms each feature to
zero mean and unit variance so that all features contribute equally to the distance metric.

### k Selection via Cross-Validation
The training set is kept strictly separate from the test set during the k sweep.
For each candidate k, a `StandardScaler → KNN` pipeline is evaluated with stratified 5-fold CV on the
training split. The k that maximises mean CV accuracy is selected, the pipeline is refit on the full
training split, and then evaluated once on the held-out test set.

### Species vs. Sex — Task Difficulty
| Aspect | Species | Sex |
|---|---|---|
| Task type | 3-class | Binary |
| Separability | High — distinct morphology | Lower — overlapping distributions |
| Typical CV optimal k | Small (1–5) | Larger (5–15) |
| Typical test accuracy | ~97–100 % | ~85–93 % |

A larger optimal k for the sex classifier reflects noisier class boundaries: a wider neighborhood averages
out local noise and generalises better than a tight single-neighbor rule would.

### Bias–Variance Tradeoff in KNN
- **Small k** → low bias, high variance (complex, wiggly boundary — overfits to local noise).
- **Large k** → high bias, low variance (smooth boundary — may underfit).
- Cross-validation locates the k that best balances these two extremes for each task.